# Chapitre 4 - Cours 2 : Thrift et Protocol Buffers, numéros de champ et évolution de schéma

Le cours 1 a montré qu'un format sans schéma doit répéter les noms de champs dans chaque
enregistrement. Apache Thrift (né chez Facebook) et Protocol Buffers (protobuf, conçu chez
Google) imposent un **schéma** et remplacent les noms par de petits **numéros de champ**
(field tags). Ce cours construit leurs encodeurs à la main, octet par octet, puis étudie
comment ces numéros organisent l'évolution du schéma. Il couvre la section *Thrift and
Protocol Buffers* et ses sous-sections *Field tags and schema evolution* et *Datatypes
and schema evolution*.

**Plan.**

1. Un schéma, une IDL et du code généré.
2. Les entiers de longueur variable : varint et zigzag.
3. L'enregistrement d'exemple en Thrift et en Protocol Buffers, octet par octet.
4. Numéros de champ et évolution du schéma : ajouter, supprimer, préserver l'inconnu.
5. Changer le type d'un champ.
6. Une matrice de compatibilité testée automatiquement.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import encoding as E
from ddia.encoding import PbField, PbMessage
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 2)
pd.set_option("display.max_colwidth", 90)
rng = np.random.default_rng(0)
R = E.EXAMPLE_RECORD
RP = E.to_pb_record(R)          # mêmes données, noms de champs à la mode protobuf